In [73]:
import warnings
import pandas as pd 
import numpy as np 
import seaborn as sns  
from sklearn import tree  
import matplotlib.pyplot as plt    
from sklearn import tree 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import  LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, roc_auc_score,ConfusionMatrixDisplay, confusion_matrix
warnings.filterwarnings("ignore")

In [74]:
df = pd.read_csv("f1_strategy_dataset_v4.csv")
df.shape

(101371, 16)

In [75]:
df.head(2)

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
0,ALB,1,MEDIUM,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.000,0.000,0,0,0.017241,0.117647,0.0
1,ALB,2,MEDIUM,1,3.0,18,93.560,Abu Dhabi Grand Prix,2023,-7.065,-7.065,0,0,0.034483,0.176471,-1.0


In [76]:
df.dtypes

Driver                        str
LapNumber                   int64
Compound                      str
Stint                       int64
TyreLife                  float64
Position                    int64
LapTime (s)               float64
Race                          str
Year                        int64
LapTime_Delta             float64
Cumulative_Degradation    float64
PitStop                     int64
PitNextLap                  int64
RaceProgress              float64
Normalized_TyreLife       float64
Position_Change           float64
dtype: object

In [77]:
df.columns

Index(['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'Race', 'Year', 'LapTime_Delta',
       'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress',
       'Normalized_TyreLife', 'Position_Change'],
      dtype='str')

In [78]:
categorical_cols = ['Driver','Compound', 'Race']

numerical_cols = ['LapNumber',  'Stint', 'TyreLife', 'Position',
       'LapTime (s)',  'Year', 'LapTime_Delta',
       'Cumulative_Degradation', 'PitStop',  'RaceProgress',
       'Normalized_TyreLife', 'Position_Change']

#### **Pipeline**

In [92]:
preprocessor = ColumnTransformer([
    ('num',StandardScaler(), numerical_cols), 
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols)
])

# pipeline 
#  Decision Tree
pipeline_dt = Pipeline(
    [
        ('prep', preprocessor), 
        ('clf', tree.DecisionTreeClassifier (class_weight='balanced', max_depth = 15))
    ]
)
#  Logistic regression
pipeline_lr = Pipeline(
    [
        ('prep', preprocessor), 
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
    ]
)

# random forest 
pipeline_rf = Pipeline(
    [
        ('prep', preprocessor), 
        ('clf', RandomForestClassifier(class_weight='balanced',  n_estimators=50, random_state=42))
    ]
)

#### **Splitting using naive approach(80-20)**

In [93]:
X = df.drop(columns = ['PitNextLap']) # features 
y = df['PitNextLap'] # target 

y = pd.DataFrame(y)

In [94]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [95]:
X_test.head(2)

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,RaceProgress,Normalized_TyreLife,Position_Change
41988,NOR,71,HARD,2,41.0,2,80.363,Mexico City Grand Prix,2024,5.356,-22.479,1,0.910256,0.525641,-1.0
13447,SAR,32,HARD,2,17.0,13,97.740,Las Vegas Grand Prix,2023,-1.017,-3.491,0,0.640000,0.485714,0.0


In [96]:
y_train.head(1)

,PitNextLap
59688,1


#### **Model Fit**

Decision Trees

In [97]:
pipeline_dt.fit(X_train, y_train)
y_pred_dt = pipeline_dt.predict(X_test)
y_prob_dt = pipeline_dt.predict_proba(X_test)[:,1]

In [98]:
# Decision Trees
print("Decision Trees")
print(classification_report(y_test, y_pred_dt))
print(roc_auc_score(y_test, y_prob_dt))

Decision Trees
              precision    recall  f1-score   support

           0       0.96      0.87      0.91     15109
           1       0.70      0.90      0.79      5166

    accuracy                           0.88     20275
   macro avg       0.83      0.88      0.85     20275
weighted avg       0.89      0.88      0.88     20275

0.9250384039869066


Logisitic regression

In [ ]:
pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)
y_prob_lr = pipeline_lr.predict_proba(X_test)[:,1]

/home/josephridge/Documents/@software-engineering/@datascience/@emobilis/@datascience/ML-Series/env/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [ ]:
# Logisitic Regression 
print("Logisitc regression")
print(classification_report(y_test, y_pred_lr))
print(roc_auc_score(y_test, y_prob_lr))

Logisitc regression
              precision    recall  f1-score   support

           0       0.89      0.72      0.80     15109
           1       0.48      0.74      0.58      5166

    accuracy                           0.73     20275
   macro avg       0.69      0.73      0.69     20275
weighted avg       0.79      0.73      0.74     20275

0.8036219806994455


Random forest 

In [ ]:
pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)
y_prob_rf = pipeline_rf.predict_proba(X_test)[:,1]

/home/josephridge/Documents/@software-engineering/@datascience/@emobilis/@datascience/ML-Series/env/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [58]:
# Random Forest
print("Random Forest")
print(classification_report(y_test, y_pred_rf))
print(roc_auc_score(y_test, y_prob_rf))

Random Forest
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     15109
           1       0.96      0.90      0.93      5166

    accuracy                           0.97     20275
   macro avg       0.97      0.94      0.95     20275
weighted avg       0.97      0.97      0.97     20275

0.9932643413213064


#### **Splitting using k-fold cross-validation**

In [67]:
df.columns

Index(['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'Race', 'Year', 'LapTime_Delta',
       'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress',
       'Normalized_TyreLife', 'Position_Change'],
      dtype='str')

In [100]:
gkf = GroupKFold(n_splits = 5)


lr_scores = cross_val_score(
    pipeline_lr, X, y, cv=gkf, groups=df['Race'],
    scoring='roc_auc', n_jobs =-1
)

print(f"# k-folds = {gkf},  Logistic Regression AUC {lr_scores.mean() } + or - {(lr_scores.std() )}")

dt_scores = cross_val_score(
    pipeline_dt, X, y, cv=gkf, groups=df['Race'],
    scoring='roc_auc', n_jobs =-1
)

print(f"# k-folds = {gkf},  Decision Trees AUC {dt_scores.mean() } + or - {(dt_scores.std() )}")

rf_scores =  cross_val_score(
    pipeline_rf, X, y, cv=gkf, groups=df['Race'],
    scoring='roc_auc', n_jobs =-1
)


print(f"# k-folds = {gkf},  RandomForest AUC {rf_scores.mean() } + or - {(rf_scores.std() )}")


# k-folds = GroupKFold(n_splits=5, random_state=None, shuffle=False),  Logistic Regression AUC 0.7638920578796323 + or - 0.007588274834934776
# k-folds = GroupKFold(n_splits=5, random_state=None, shuffle=False),  Decision Trees AUC 0.7891299057020349 + or - 0.008123485334355734
# k-folds = GroupKFold(n_splits=5, random_state=None, shuffle=False),  RandomForest AUC0.8767838894417255 + or - 0.00968107682404144


Summary: 

- The AUC of Logistic regression is at 76.389% with a spread of + or - 0.0076
- The AUC of Random Forest is at 87.678% with a spread of + or - 0.0096
  
=> for LR, it ranks pit stop lpa higher than no pit stop laps by 76.389%
=> for RF, it ranks pit stop lpa higher than no pit stop laps by 87.678%

In [71]:
# gkf = GroupKFold(n_splits = 10)


# lr_scores = cross_val_score(
#     pipeline_lr, X, y, cv=gkf, groups=df['Race'],
#     scoring='roc_auc', n_jobs =-1
# )

# print(f"# k-folds = {gkf},  Logistic Regression AUC {lr_scores.mean() } + or - {(lr_scores.std() )}")

# rf_scores =  cross_val_score(
#     pipeline_rf, X, y, cv=gkf, groups=df['Race'],
#     scoring='roc_auc', n_jobs =-1
# )


# print(f"# k-folds = {gkf},  RandomForest AUC{rf_scores.mean() } + or - {(rf_scores.std() )}")


# k-folds = GroupKFold(n_splits=10, random_state=None, shuffle=False),  Logistic Regression AUC 0.7609275359973569 + or - 0.044036722647474984
# k-folds = GroupKFold(n_splits=10, random_state=None, shuffle=False),  RandomForest AUC0.8786451807822437 + or - 0.02423233545667786
